In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# トランスパイラー設定の比較

<Accordion>
<AccordionItem title="Package versions">

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
異なるトランスパイラー設定は、古典処理時間が長くなる代わりに、回路にさまざまな最適化を提供します。このガイドでは、各種設定のパフォーマンスをテストするために、回路の作成・トランスパイル・送信の全プロセスを順を追って説明します。

同じ設定でも、ある回路の結果を改善する一方で別の回路を悪化させる場合があることに注意してください。実際のハードウェアで実行する前に、トランスパイル後の回路を必ず確認してください。

## セットアップとサンプル回路の作成

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

トランスパイラーが最適化を試みるための小さな回路を作成します。この例では、状態`111`をマークするオラクルを用いてGroverのアルゴリズムを実行する回路を作成します。次に、後で比較するために理想的な分布（完全な量子コンピューターで無限回実行した場合に期待される測定値）をシミュレーションします。

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg)

## トランスパイル
次に、QPU向けに回路をトランスパイルします。`optimization_level`を`0`（最低）に設定した場合と`3`（最高）に設定した場合のトランスパイラーのパフォーマンスを比較します。最低の最適化レベルは、デバイス上で回路を実行するために必要な最低限の処理を行います。具体的には、回路のQubitをデバイスのQubitにマッピングし、すべての2Qubit演算を可能にするためにSWAPゲートを追加します。最高の最適化レベルはより高度で、全体的なゲート数を削減するためにさまざまな工夫を活用します。マルチQubitゲートはエラーレートが高く、Qubitは時間とともにデコヒーレンスするため、短い回路の方がより良い結果をもたらすはずです。

> **Important:** この例はIBM Quantum&reg;ハードウェアを使用していますが、Qiskit互換の任意のQPUで試すことができます。結果は異なる場合があります。

次のセルでは、`optimization_level`の両方の値について`qc`をトランスパイルし、2Qubitゲートの数を出力し、トランスパイルされた回路をリストに追加します。トランスパイラーのアルゴリズムの一部はランダム化されているため、再現性のためにシードを設定します。

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

CNOTはエラーレートが高いため、`optimization_level=3`でトランスパイルされた回路の方がはるかに優れたパフォーマンスを発揮するはずです。

パフォーマンスをさらに向上させる別の方法として、[ダイナミカルデカップリング](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling)があります。これは、アイドル状態のQubitにゲートのシーケンスを適用することで、環境との望ましくない相互作用の一部を打ち消します。次のセルでは、`optimization_level=3`でトランスパイルされた回路にダイナミカルデカップリングを追加し、リストに追加します。

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

## View results

Finally, plot the results from the device runs against the ideal distribution. You can see the results with `optimization_level=3` are closer to the ideal distribution due to the lower gate count, and `optimization_level=3 + dd` is even closer due to the dynamical decoupling.

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg)

## 回路を実行する
この時点で、異なる設定でトランスパイルされた回路のリストが揃っています。次に、SamplerプリミティブでこれらのCircuitを実行し、結果を`result`に格納します。

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


## 結果を表示する
最後に、デバイス実行の結果を理想的な分布と比較してプロットします。ゲート数が少ないため`optimization_level=3`の結果が理想的な分布に近く、ダイナミカルデカップリングの効果により`optimization_level=3 + dd`がさらに近いことが確認できます。